# Ethanol IR walkthrough (≈5–10 min)

A short **analysis-path** tour of ChemSpec / `spectrum_core` using a **public-domain** measured IR spectrum (PNNL via NIST Chemistry WebBook).

**What you will do**

1. Load `fixtures/public/ethanol_ir_pnnl.jdx`
2. Note license / source (`SOURCES.md`)
3. Plot the spectrum (matplotlib)
4. Apply baseline (+ optional smooth)
5. Find peaks with **FWHM** and **area**
6. Optionally export a peaks CSV and mention session save

**Setup** (repo root):

```bash
pip install -e ".[dev,ui,baselines]"
jupyter notebook examples/ethanol_ir_walkthrough.ipynb
# or: jupyter lab …
```

Matplotlib only — **NiceGUI is not required**.

---

### Disclaimer — no compound identification

ChemSpec Workbench and `spectrum_core` **do not identify compounds**.  
The fixture filename and NIST title say “Ethanol” for **provenance only**. Peak centers, heights, FWHM, and area are **geometric metrics** on the loaded trace — not a library match or chemical ID claim.


## 0 · Imports & locate the fixture

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from spectrum_core import (
    apply_step,
    find_peaks,
    ingest,
    peaks_to_csv,
    save_session,
)

FIXTURE_REL = Path("fixtures") / "public" / "ethanol_ir_pnnl.jdx"
SOURCES_REL = Path("fixtures") / "public" / "SOURCES.md"


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / FIXTURE_REL).is_file():
            return p
    raise FileNotFoundError(
        f"Could not find {FIXTURE_REL}. Open the notebook from the "
        "chemspec-workbench repo (or set the working directory to the repo root)."
    )


ROOT = find_repo_root()
FIXTURE = ROOT / FIXTURE_REL
SOURCES = ROOT / SOURCES_REL
print("repo root:", ROOT)
print("fixture  :", FIXTURE)
print("sources  :", SOURCES if SOURCES.is_file() else "(missing)")


## 1 · License / source (read this)

The JCAMP is redistributed from the [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/) (SRD 69). The file header / page lists **Owner: Public domain**; origin is Pacific Northwest National Laboratory under an IARPA contract.

Full attribution, NIST disclaimer, and download URLs live in:

`fixtures/public/SOURCES.md`

**Do not** treat the NIST compound title as a ChemSpec identification result.


In [ ]:
# Peek at the attribution file (first section)
print(SOURCES.read_text(encoding="utf-8").split("---")[0])
print("… see", SOURCES_REL.as_posix(), "for full tables and NIST disclaimer.")


## 2 · Load the public ethanol IR JCAMP

`ingest()` dispatches `.jdx` / `.dx` to basic JCAMP-DX ingest (`jcamp`).  
PNNL n/k composites may emit an X-Check warning from the parser; ingest still exposes a usable absorption-index intensity trace (see `SOURCES.md` notes).


In [ ]:
raw = ingest(FIXTURE)

print("title   :", raw.title, "  ← NIST label (provenance only)")
print("points  :", len(raw))
print("x_unit  :", raw.x_unit, "  y_unit:", raw.y_unit)
print("x range :", float(np.nanmin(raw.x)), "…", float(np.nanmax(raw.x)))
print("y range :", float(np.nanmin(raw.y)), "…", float(np.nanmax(raw.y)))


## 3 · Plot the raw spectrum

IR plots conventionally show **descending** wavenumber (cm⁻¹) on the x-axis.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(raw.x, raw.y, color="C0", lw=0.9)
ax.set_xlabel("Wavenumber (cm⁻¹)")
ax.set_ylabel("Intensity (as ingested)")
ax.set_title("Public ethanol IR fixture — raw (not compound ID)")
ax.invert_xaxis()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 4 · Baseline (+ optional smooth)

Use the **processing pipeline** so steps are recorded in an append-only history (handy for session save later). Polynomial baseline always works without extras; optional `asls` / `mpls` need `pip install -e ".[baselines]"`.


In [ ]:
working, history = apply_step(
    raw, None, "baseline", {"method": "polynomial", "degree": 2}
)
working, history = apply_step(
    working, history, "smooth", {"window_length": 11, "polyorder": 3}
)

print("pipeline steps:")
for line in history.summary_lines():
    print(" ", line)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(raw.x, raw.y, color="0.65", lw=0.8, label="raw")
ax.plot(working.x, working.y, color="C0", lw=1.2, label="baseline + smooth")
ax.set_xlabel("Wavenumber (cm⁻¹)")
ax.set_ylabel("Intensity (as ingested)")
ax.set_title("Baseline + Savitzky–Golay smooth")
ax.invert_xaxis()
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 5 · Find peaks (FWHM + area)

`find_peaks` wraps `scipy.signal.find_peaks` and adds **FWHM** and **area** from prominence-relative half-max bounds (see `spectrum_core.peaks` docstring). These are **not** band assignments.


In [ ]:
PROMINENCE = 0.01  # tweak if you want fewer / more peaks
peaks = find_peaks(working, prominence=PROMINENCE)
print(f"n_peaks = {len(peaks)}  (prominence={PROMINENCE})")
print()
print(f"{'idx':>6}  {'x':>10}  {'y':>12}  {'prominence':>12}  {'FWHM':>12}  {'area':>12}")
print("-" * 72)
for p in peaks:
    fwhm_s = f"{p.fwhm:12.4f}" if np.isfinite(p.fwhm) else f"{'nan':>12}"
    area_s = f"{p.area:12.4f}" if np.isfinite(p.area) else f"{'nan':>12}"
    print(
        f"{p.index:6d}  {p.x:10.1f}  {p.y:12.6g}  {p.prominence:12.6g}  "
        f"{fwhm_s}  {area_s}"
    )


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(working.x, working.y, color="C0", lw=1.1, label="processed")
if peaks:
    ax.scatter(
        [p.x for p in peaks],
        [p.y for p in peaks],
        color="C3",
        s=32,
        zorder=5,
        label=f"peaks (n={len(peaks)})",
    )
    for p in peaks[:8]:
        ax.annotate(
            f"{p.x:.0f}",
            (p.x, p.y),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
        )
ax.set_xlabel("Wavenumber (cm⁻¹)")
ax.set_ylabel("Intensity (as ingested)")
ax.set_title("Peaks with labels (wavenumber only — not band ID)")
ax.invert_xaxis()
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 6 · Optional export — peaks CSV & session save

- **`peaks_to_csv`** — peak table (`index`, `x`, `y`, `prominence`, `fwhm`, `area`)
- **`save_session`** — analysis snapshot (`.csw.json`): embedded spectrum + pipeline history + peaks + notes

Both are **analysis artifacts**, not identification reports.


In [ ]:
# Write next to the notebook working directory (safe to delete)
OUT = Path("ethanol_ir_walkthrough_out")
OUT.mkdir(exist_ok=True)

csv_path = OUT / "ethanol_ir_peaks.csv"
peaks_to_csv(peaks, csv_path)
print("peaks CSV →", csv_path.resolve())
print(csv_path.read_text(encoding="utf-8").splitlines()[0])
print("…", len(peaks), "data rows")

session_path = OUT / "ethanol_ir_walkthrough.csw.json"
save_session(
    session_path,
    raw,
    history=history,
    peaks=peaks,
    notes=(
        "Ethanol IR tutorial walkthrough. "
        "Analysis snapshot — not compound identification. "
        f"Attribution: {SOURCES_REL.as_posix()}"
    ),
    source_path=str(FIXTURE),
    processing={
        "baseline_on": True,
        "baseline_method": "polynomial",
        "baseline_degree": 2,
        "prominence": PROMINENCE,
    },
)
print("session   →", session_path.resolve())


## Done

You exercised a real ChemSpec analysis path on a **public measured IR** fixture:

| Step | API |
|------|-----|
| Load JCAMP | `ingest` / `ingest_jcamp` |
| Baseline + smooth | `apply_step` (`baseline`, `smooth`) |
| Peaks + FWHM/area | `find_peaks` |
| Export | `peaks_to_csv`, `save_session` |

**Still true:** ChemSpec does **not** identify ethanol (or any compound). For attribution see [`fixtures/public/SOURCES.md`](../fixtures/public/SOURCES.md).

Script twin (headless / CI-friendly): `python examples/ethanol_ir_walkthrough.py`
